# Member 3 — Differential Gene Expression & Figure 4d Reproduction

**Paper**: Janesick et al. (2023), *Nature Communications* — "High resolution mapping of the tumor microenvironment using integrated single-cell, spatial and in situ analysis"

**Goal**: Load Member 1's processed + annotated scFFPE-seq AnnData object and reproduce:
- **Figure 4d** — Dot plot of canonical markers and differentially expressed genes across DCIS #1, DCIS #2, and Invasive Tumor ROIs
- Supporting DGE analysis between the three tumor subtypes

**Input**: `annotated_adata.h5ad` from Member 1 (shared via Google Drive)  
**Output**: `figures/figure4d_dotplot.png`, `figures/figure4d_supplementary.png`

---
### About Figure 4d
Figure 4d shows a dot plot where:
- **Rows** = cell type groups (DCIS #1, DCIS #2, Invasive Tumor — plus supporting cell types)
- **Columns** = marker genes grouped by cell type category (Tumor, Myoepithelial, Stromal, T Cells, B Cells, Macrophage, Endothelial)
- **Dot size** = fraction of cells in that group expressing the gene
- **Dot color** = mean expression level in that group

Key biological insight: MZB1 marks DCIS #1 exclusively; KRT15/KRT23/ALDH1A3 are high in DCIS #1 myoepithelial cells but reduced in DCIS #2; MMP12 is absent in the invasive ROI.

## Step 1 — Install Dependencies

In [ ]:
# Run this once — restart runtime after if prompted
!pip install -q scanpy matplotlib seaborn openpyxl anndata

## Step 2 — Imports & Settings

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
sc.set_figure_params(dpi=120, facecolor='white', figsize=(10, 6))
sc.settings.verbosity = 1

os.makedirs('figures', exist_ok=True)
print("Imports OK. Scanpy version:", sc.__version__)

## Step 3 — Mount Google Drive & Load Member 1's AnnData

> **Coordinate with Member 1**: They should save their output file to a shared Drive folder, e.g. `MyDrive/tumor_project/annotated_adata.h5ad`.  
> Update the `H5AD_PATH` variable below to match wherever they placed it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── UPDATE THIS PATH to wherever Member 1 saved the file ──
H5AD_PATH = '/content/drive/MyDrive/tumor_project/annotated_adata.h5ad'
# ──────────────────────────────────────────────────────────

print(f"Looking for file at: {H5AD_PATH}")
print(f"File exists: {os.path.exists(H5AD_PATH)}")

In [ ]:
# Load the AnnData — 1 GB file, expect ~30-60 seconds
print("Loading annotated_adata.h5ad ... (may take a minute)")
adata = sc.read_h5ad(H5AD_PATH)

print(f"\nLoaded successfully!")
print(f"  Cells : {adata.n_obs:,}")
print(f"  Genes : {adata.n_vars:,}")
print(f"  Obs columns : {list(adata.obs.columns)}")
print()
print(adata)

## Step 4 — Inspect Annotations from Member 1

We need to confirm which column holds cell-type annotations, and that the expected cell types (DCIS #1, DCIS #2, Invasive Tumor, etc.) are present.

In [ ]:
# ── Identify the cell-type annotation column ──
# Member 1 likely saved it as 'cell_type', 'leiden', 'annotation', or 'cluster'
# Check all columns:
print("All obs columns and their unique value counts:")
for col in adata.obs.columns:
    n = adata.obs[col].nunique()
    dtype = adata.obs[col].dtype
    print(f"  {col:30s}  ({dtype})  ->  {n} unique values")

In [ ]:
# ── UPDATE THIS if the annotation column has a different name ──
ANNOT_COL = 'cell_type'   # change if needed (e.g. 'annotation', 'leiden')
# ──────────────────────────────────────────────────────────────

if ANNOT_COL not in adata.obs.columns:
    raise ValueError(f"Column '{ANNOT_COL}' not found. Update ANNOT_COL above.")

print(f"Using annotation column: '{ANNOT_COL}'")
print()
print("Cell type counts:")
print(adata.obs[ANNOT_COL].value_counts().to_string())

## Step 5 — Ensure Log-Normalized Expression

The dot plot needs log-normalized counts in `adata.X`. If Member 1 already stored raw counts in `adata.layers['counts']` and has normalized `adata.X`, we re-use that. Otherwise we normalize here.

In [ ]:
import scipy.sparse as sp

# Check if X looks like it's already log-normalized
if sp.issparse(adata.X):
    sample_max = adata.X[:500, :].max()
else:
    sample_max = adata.X[:500, :].max()

print(f"Max value in X (sample): {sample_max:.2f}")

if sample_max > 50:  # likely raw counts
    print("X appears to be raw counts → normalizing now")
    if 'counts' not in adata.layers:
        adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print(f"After normalization, max value: {adata.X[:500,:].max():.2f}")
else:
    print("X appears already log-normalized — no action needed.")

## Step 6 — Define Marker Genes for Figure 4d

These are the exact gene groups shown in Figure 4d of Janesick et al. 2023.  
Grouped by cell-type category (columns of the dot plot).

In [ ]:
# Exact marker genes from Figure 4d, grouped by cell-type category
MARKER_GENES = {
    'Tumor'         : ['FASN', 'FDX1A', 'GATA3', 'AGR3', 'SERPINA3',
                       'TACSTD2', 'ABCC5', 'MKI67'],
    'Myoepithelial' : ['KRT23', 'ALDH1A3', 'KRT15', 'MYLK', 'ACTA2'],
    'Stromal'       : ['GJB2', 'SFRP2', 'POSTN', 'MYH11'],
    'T Cells'       : ['CXCR4', 'CD8A', 'TRAC', 'CD4'],
    'B Cells'       : ['MS4A1', 'BANK1', 'MZB1'],
    'Macrophage'    : ['C1QA', 'C1SerH48', 'MMP12', 'ITGAX'],
    'Endothelial'   : ['LINC15', 'AQP1', 'VWF', 'PECAM1']
}

# Check what's available in this AnnData
all_genes = [g for grp in MARKER_GENES.values() for g in grp]
found   = [g for g in all_genes if g in adata.var_names]
missing = [g for g in all_genes if g not in adata.var_names]

print(f"Total marker genes requested : {len(all_genes)}")
print(f"Found in AnnData             : {len(found)}")
print(f"Missing                      : {missing if missing else 'None'}")

# Remove missing genes from groups
MARKER_GENES_FILTERED = {}
for grp, genes in MARKER_GENES.items():
    present = [g for g in genes if g in adata.var_names]
    if present:
        MARKER_GENES_FILTERED[grp] = present

print(f"\nFiltered groups:")
for k, v in MARKER_GENES_FILTERED.items():
    print(f"  {k:15s}: {v}")

## Step 7 — Map & Reconcile Cell Type Names

Member 1 may have used slightly different annotation strings. This cell maps them to the paper's canonical names.

In [ ]:
# ── Flexible name-mapping: adjust right-hand side if Member 1 used different labels ──
# Format: 'canonical_paper_name': ['possible_names_in_adata', ...]
NAME_MAP = {
    'DCIS #1'               : ['DCIS #1', 'DCIS1', 'dcis1', 'DCIS_1'],
    'DCIS #2'               : ['DCIS #2', 'DCIS2', 'dcis2', 'DCIS_2'],
    'Invasive Tumor'        : ['Invasive Tumor', 'Invasive', 'invasive_tumor',
                               'Proliferative Invasive', 'Invasive carcinoma'],
    'ACTA2+ Myoepithelial'  : ['ACTA2+ Myoepithelial', 'ACTA2+_Myoepithelial',
                               'Myoepithelial', 'myoepithelial', 'ACTA2'],
    'KRT15+ Myoepithelial'  : ['KRT15+ Myoepithelial', 'KRT15+_Myoepithelial', 'KRT15'],
    'Stromal'               : ['Stromal', 'stromal'],
    'T Cells'               : ['T Cells', 'T_Cells', 'CD8+ T Cells', 'CD4+ T Cells'],
    'B Cells'               : ['B Cells', 'B_Cells'],
    'Macrophages 1'         : ['Macrophages 1', 'Macrophages_1', 'Macrophages'],
    'Macrophages 2'         : ['Macrophages 2', 'Macrophages_2'],
    'Endothelial'           : ['Endothelial', 'endothelial'],
}

existing_types = set(adata.obs[ANNOT_COL].unique())

# Build reverse lookup: adata_label -> canonical_label
reverse_map = {}
for canonical, variants in NAME_MAP.items():
    for v in variants:
        if v in existing_types:
            reverse_map[v] = canonical

# Apply mapping
adata.obs['cell_type_canonical'] = (
    adata.obs[ANNOT_COL]
    .map(reverse_map)
    .fillna(adata.obs[ANNOT_COL])   # keep original if no mapping found
)

print("Mapping applied. Canonical cell types in data:")
print(adata.obs['cell_type_canonical'].value_counts().to_string())

In [ ]:
# Cell types to include in the Figure 4d dot plot
# Ordered to match the paper's row order (top to bottom)
ROW_ORDER = [
    'DCIS #1',
    'DCIS #2',
    'Invasive Tumor',
    'ACTA2+ Myoepithelial',
    'KRT15+ Myoepithelial',
    'Stromal',
    'T Cells',
    'B Cells',
    'Macrophages 1',
    'Macrophages 2',
    'Endothelial',
]

available_rows = [r for r in ROW_ORDER
                  if r in adata.obs['cell_type_canonical'].values]
missing_rows   = [r for r in ROW_ORDER if r not in available_rows]

print(f"Rows available for plotting : {available_rows}")
print(f"Rows not found in data      : {missing_rows}")

adata_sub = adata[adata.obs['cell_type_canonical'].isin(available_rows)].copy()
# Set category order
adata_sub.obs['cell_type_canonical'] = pd.Categorical(
    adata_sub.obs['cell_type_canonical'],
    categories=available_rows,
    ordered=True
)

print(f"\nCells in subset: {adata_sub.n_obs:,}")

## Step 8 — Reproduce Figure 4d: Dot Plot

In [ ]:
# Scanpy dot plot — replicates Figure 4d
dp = sc.pl.dotplot(
    adata_sub,
    var_names=MARKER_GENES_FILTERED,
    groupby='cell_type_canonical',
    dendrogram=False,
    standard_scale='var',         # scale each gene 0→1 across groups (matches paper)
    color_map='RdBu_r',           # matches paper's warm/cool colormap
    dot_max=0.8,
    colorbar_title='Mean\nexpression\nin group',
    size_title='Fraction of cells\nin group (%)',
    figsize=(18, 6),
    return_fig=True,
    show=False,
)

# Add title and tidy up
dp.fig.suptitle(
    'Figure 4d Reproduction — Canonical Markers & DGE across Tumor ROIs\n'
    'Janesick et al. (2023) Nature Communications',
    fontsize=12, y=1.03
)

plt.savefig('figures/figure4d_dotplot.png', dpi=150, bbox_inches='tight')
dp.show()
print("\nFigure 4d saved to figures/figure4d_dotplot.png")

## Step 9 — DGE: Tumor Subtypes (DCIS #1 vs DCIS #2 vs Invasive)

Reproduces the biological comparisons underlying Figure 4d. We use the Wilcoxon rank-sum test (scanpy default) to find genes differentially expressed between the three tumor subtypes.

In [ ]:
# Subset to tumor cell types only for DGE
tumor_types = ['DCIS #1', 'DCIS #2', 'Invasive Tumor']
tumor_available = [t for t in tumor_types
                   if t in adata.obs['cell_type_canonical'].values]

adata_tumor = adata[adata.obs['cell_type_canonical'].isin(tumor_available)].copy()
adata_tumor.obs['cell_type_canonical'] = pd.Categorical(
    adata_tumor.obs['cell_type_canonical'],
    categories=tumor_available
)

print(f"Tumor cells for DGE:")
print(adata_tumor.obs['cell_type_canonical'].value_counts())

In [ ]:
# Run DGE — Wilcoxon test, one-vs-rest per group
sc.tl.rank_genes_groups(
    adata_tumor,
    groupby='cell_type_canonical',
    method='wilcoxon',
    use_raw=False,
    n_genes=50,
    pts=True,          # include % expressed
)

print("DGE complete. Top 10 genes per group:")
sc.pl.rank_genes_groups(adata_tumor, n_genes=10, sharey=False, show=True)

In [ ]:
# Extract results as a tidy dataframe for inspection
dge_results = {}
for group in tumor_available:
    df = sc.get.rank_genes_groups_df(
        adata_tumor,
        group=group,
        pval_cutoff=0.05,
        log2fc_min=1.0
    )
    dge_results[group] = df
    print(f"\nTop 15 upregulated genes in {group}:")
    print(df.head(15)[['names', 'logfoldchanges', 'pvals_adj', 'pct_nz_group']].to_string(index=False))

In [ ]:
# Check key paper findings:
# - MZB1 should be exclusive to DCIS #1
# - MMP12 should be absent/low in Invasive ROI
# - GJB2 stromal cells in DCIS #2
# - ALDH1A3, KRT15, KRT23 high in DCIS #1 myoepithelial

KEY_GENES = ['MZB1', 'MMP12', 'GJB2', 'ALDH1A3', 'KRT15', 'KRT23', 'AGR3']
KEY_FOUND = [g for g in KEY_GENES if g in adata_tumor.var_names]

if KEY_FOUND:
    print("Spot-check of key paper findings (mean log-norm expression per group):")
    import scipy.sparse as sp
    X = adata_tumor[:, KEY_FOUND].X
    if sp.issparse(X):
        X = X.toarray()
    summary = pd.DataFrame(
        X,
        index=adata_tumor.obs['cell_type_canonical'],
        columns=KEY_FOUND
    ).groupby(level=0).mean().round(3)
    print(summary.to_string())
else:
    print("Key genes not in this AnnData — check gene names.")

## Step 10 — Supplementary: DGE Heatmap of Top Genes

In [ ]:
# Visualize top 8 genes per tumor subtype as a heatmap
sc.pl.rank_genes_groups_heatmap(
    adata_tumor,
    n_genes=8,
    groupby='cell_type_canonical',
    show_gene_labels=True,
    figsize=(14, 5),
    show=False
)
plt.suptitle(
    'Supplementary: Top DGE Genes per Tumor Subtype (Wilcoxon, one-vs-rest)\n'
    'DCIS #1 / DCIS #2 / Invasive Tumor',
    fontsize=11, y=1.05
)
plt.savefig('figures/figure4d_supplementary_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Supplementary heatmap saved.")

## Step 11 — Violin Plots for Specific Paper Findings

These replicate the biological claims in the paper text accompanying Figure 4.

In [ ]:
# Genes highlighted in paper text: MZB1 (DCIS #1 exclusive), MMP12 (absent in invasive)
VIOLIN_GENES = [g for g in ['MZB1', 'MMP12', 'GJB2', 'ALDH1A3', 'KRT15', 'AGR3']
                if g in adata_tumor.var_names]

if VIOLIN_GENES:
    sc.pl.violin(
        adata_tumor,
        keys=VIOLIN_GENES,
        groupby='cell_type_canonical',
        rotation=30,
        show=False
    )
    plt.suptitle(
        'Violin: Key Differentially Expressed Genes — Tumor Subtypes',
        fontsize=11, y=1.02
    )
    plt.savefig('figures/figure4d_violin_key_genes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Violin plot saved.")
else:
    print("None of the key violin genes were found in this AnnData.")

## Step 12 — Summary Statistics

In [ ]:
print("=" * 55)
print("MEMBER 3 ANALYSIS SUMMARY")
print("=" * 55)
print(f"Input file       : {H5AD_PATH}")
print(f"Total cells      : {adata.n_obs:,}")
print(f"Total genes      : {adata.n_vars:,}")
print()
print("Cells per tumor subtype used for Figure 4d:")
for t in tumor_available:
    n = (adata.obs['cell_type_canonical'] == t).sum()
    print(f"  {t:25s}: {n:,} cells")
print()
print("Paper reports (from Figure 4 and Methods):")
print("  DCIS #1 ROI has highest ACTA2+ myoepithelial cells")
print("  DCIS #2 ROI has invasive cells present")
print("  Invasive ROI: myoepithelial cells completely absent")
print("  MZB1: exclusive marker of DCIS #1")
print("  MMP12: absent from invasive ROI macrophages")
print()
print("Figures saved to ./figures/:")
print("  figure4d_dotplot.png")
print("  figure4d_supplementary_heatmap.png")
print("  figure4d_violin_key_genes.png")

## Step 13 — Save DGE Results & Upload Figures to Drive

In [ ]:
# Save DGE tables as CSVs
os.makedirs('dge_results', exist_ok=True)

for group, df in dge_results.items():
    safe_name = group.replace(' ', '_').replace('#', '')
    path = f'dge_results/dge_{safe_name}.csv'
    df.to_csv(path, index=False)
    print(f"Saved: {path} ({len(df)} genes)")

print("\nAll DGE tables saved.")

In [ ]:
# Copy figures and DGE tables back to shared Drive folder
import shutil

DRIVE_OUTPUT = '/content/drive/MyDrive/tumor_project/member3_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy figures
for f in os.listdir('figures'):
    shutil.copy(f'figures/{f}', f'{DRIVE_OUTPUT}/{f}')
    print(f"Copied: figures/{f} -> Drive")

# Copy DGE CSVs
for f in os.listdir('dge_results'):
    shutil.copy(f'dge_results/{f}', f'{DRIVE_OUTPUT}/{f}')
    print(f"Copied: dge_results/{f} -> Drive")

print(f"\nAll outputs saved to: {DRIVE_OUTPUT}")

---
## Troubleshooting Notes

| Problem | Likely Cause | Fix |
|---------|-------------|-----|
| `KeyError: 'cell_type'` | Column name differs | Update `ANNOT_COL` in Step 4 |
| Cell types not found | Labels differ from paper | Update `NAME_MAP` in Step 7 |
| Many genes missing | adata only has 313 Xenium genes | Confirm Member 1 used the whole-transcriptome scFFPE-seq matrix |
| Drive mount fails | Not signed into right Google account | Re-run `drive.mount(force_remount=True)` |
| `MemoryError` | Large matrix | Use `adata_sub` (subset) for all plots |
| Dot plot all grey | Expression not log-normalized | Re-run Step 5 with `sample_max` check |

### How to confirm you reproduced Figure 4d correctly
Compare your dot plot against Figure 4d in the paper:
1. **DCIS #1 row** should show high expression in Myoepithelial genes (KRT23, ALDH1A3, KRT15) and B cell marker MZB1
2. **DCIS #2 row** should have GJB2 stromal signal and less myoepithelial expression than DCIS #1
3. **Invasive Tumor row** should show strong FASN/CDH2 expression, absent myoepithelial markers, and absent MMP12